# Multifractal Analysis of Neural Dynamics

Advanced fractal analysis revealing multiscale temporal structure.

**Methods:**
- Multifractal Detrended Fluctuation Analysis (MF-DFA)
- Wavelet Transform Modulus Maxima (WTMM)
- Singularity spectrum f(α)
- Generalized Hurst exponents h(q)
- Multifractal width as measure of complexity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from neuros_mechint.fractals import (
    MultifractalDetrendedFluctuationAnalysis,
    WaveletMultifractal
)
from neuros_mechint.visualization import MultifractalVisualizer

%matplotlib inline

## 1. Generate Test Signals

Create monofractal and multifractal time series for comparison.

In [ ]:
def generate_fbm(n, H):
    """Generate fractional Brownian motion (monofractal)."""
    fgn = np.random.randn(n)
    
    # Apply power-law filter in Fourier domain
    f = np.fft.rfftfreq(n)[1:]
    spectrum = np.zeros(n//2 + 1)
    spectrum[0] = 0
    spectrum[1:] = f**(-H - 0.5)
    
    fgn_fft = np.fft.rfft(fgn)
    fgn_fft *= np.sqrt(spectrum)
    
    fbm = np.fft.irfft(fgn_fft, n)
    return np.cumsum(fbm)

def generate_multifractal_cascade(n):
    """Generate multifractal cascade (multiplicative cascade)."""
    # Start with uniform signal
    signal = np.ones(n)
    
    # Multiplicative cascade
    scale = n
    while scale > 2:
        for i in range(0, n, scale):
            # Split into two parts with random weights
            w1 = np.random.uniform(0.3, 0.7)
            w2 = 1 - w1
            signal[i:i+scale//2] *= w1 * 2
            signal[i+scale//2:i+scale] *= w2 * 2
        scale //= 2
    
    return np.cumsum(signal - signal.mean())

# Generate signals
n = 4096
monofractal = generate_fbm(n, H=0.7)  # Persistent (H > 0.5)
multifractal = generate_multifractal_cascade(n)

# Visualize
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(monofractal, 'b-', linewidth=0.5)
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Monofractal Signal (FBM, H=0.7)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(multifractal, 'r-', linewidth=0.5)
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Multifractal Signal (Cascade)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Multifractal DFA (MF-DFA)

Compute fluctuation functions F_q(s) for different moments q.

In [ ]:
# Setup MF-DFA
q_values = np.arange(-5, 6, 1)
scales = np.logspace(1, 3, 20).astype(int)
scales = np.unique(scales)

mfdfa_mono = MultifractalDetrendedFluctuationAnalysis(q_values=q_values, scales=scales)
mfdfa_multi = MultifractalDetrendedFluctuationAnalysis(q_values=q_values, scales=scales)

# Analyze both signals
result_mono = mfdfa_mono.analyze(monofractal)
result_multi = mfdfa_multi.analyze(multifractal)

# Visualize fluctuation functions
viz = MultifractalVisualizer()

fig_mono = viz.plot_mfdfa_results(
    scales=scales,
    fluctuations=result_mono['fluctuations'],
    q_values=q_values,
    title="MF-DFA: Monofractal Signal"
)
fig_mono.show()

fig_multi = viz.plot_mfdfa_results(
    scales=scales,
    fluctuations=result_multi['fluctuations'],
    q_values=q_values,
    title="MF-DFA: Multifractal Signal"
)
fig_multi.show()

## 3. Generalized Hurst Exponents

h(q) characterizes scaling for different moments.

In [ ]:
# Extract Hurst exponents
h_mono = result_mono['hurst_exponents']
h_multi = result_multi['hurst_exponents']

# Compute tau(q) = qh(q) - 1
tau_mono = q_values * h_mono - 1
tau_multi = q_values * h_multi - 1

# Visualize
fig_mono_h = viz.plot_scaling_exponents(
    q_values=q_values,
    tau_q=tau_mono,
    h_q=h_mono,
    title="Scaling Exponents: Monofractal"
)
fig_mono_h.show()

fig_multi_h = viz.plot_scaling_exponents(
    q_values=q_values,
    tau_q=tau_multi,
    h_q=h_multi,
    title="Scaling Exponents: Multifractal"
)
fig_multi_h.show()

# Comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(q_values, h_mono, 'bo-', linewidth=2, markersize=8, label='Monofractal')
plt.plot(q_values, h_multi, 'rs-', linewidth=2, markersize=8, label='Multifractal')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random walk')
plt.xlabel('Moment q')
plt.ylabel('h(q)')
plt.title('Generalized Hurst Exponent')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Multifractal width
width_mono = h_mono.max() - h_mono.min()
width_multi = h_multi.max() - h_multi.min()
plt.bar(['Monofractal', 'Multifractal'], [width_mono, width_multi], 
        color=['blue', 'red'], alpha=0.7)
plt.ylabel('Multifractal Width Δh')
plt.title('Multifractality Strength')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nMultifractal Characterization:")
print(f"Monofractal:")
print(f"  h(2) = {h_mono[q_values == 2][0]:.3f}  (standard Hurst)")
print(f"  Δh = {width_mono:.3f}  (width, small for monofractal)")
print(f"\nMultifractal:")
print(f"  h(2) = {h_multi[q_values == 2][0]:.3f}")
print(f"  Δh = {width_multi:.3f}  (width, large for multifractal)")

## 4. Singularity Spectrum f(α)

The spectrum f(α) describes the distribution of local Hölder exponents.

In [ ]:
# Compute singularity spectrum via Legendre transform
# α = dτ/dq, f(α) = qα - τ(q)

def compute_singularity_spectrum(q_values, tau_q):
    """Compute singularity spectrum from τ(q)."""
    # Numerical derivative
    dq = q_values[1] - q_values[0]
    alpha = np.gradient(tau_q, dq)
    f_alpha = q_values * alpha - tau_q
    return alpha, f_alpha

alpha_mono, f_alpha_mono = compute_singularity_spectrum(q_values, tau_mono)
alpha_multi, f_alpha_multi = compute_singularity_spectrum(q_values, tau_multi)

# Visualize
fig_spec_mono = viz.plot_singularity_spectrum(
    alpha=alpha_mono,
    f_alpha=f_alpha_mono,
    title="Singularity Spectrum: Monofractal"
)
fig_spec_mono.show()

fig_spec_multi = viz.plot_singularity_spectrum(
    alpha=alpha_multi,
    f_alpha=f_alpha_multi,
    title="Singularity Spectrum: Multifractal"
)
fig_spec_multi.show()

# Compare spectra
plt.figure(figsize=(10, 6))
plt.plot(alpha_mono, f_alpha_mono, 'bo-', linewidth=2, markersize=8, label='Monofractal')
plt.plot(alpha_multi, f_alpha_multi, 'rs-', linewidth=2, markersize=8, label='Multifractal')
plt.xlabel('Hölder Exponent α')
plt.ylabel('Fractal Dimension f(α)')
plt.title('Singularity Spectrum Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Spectrum properties
width_alpha_mono = alpha_mono.max() - alpha_mono.min()
width_alpha_multi = alpha_multi.max() - alpha_multi.min()

print(f"\nSingularity Spectrum Width:")
print(f"Monofractal: Δα = {width_alpha_mono:.3f}")
print(f"Multifractal: Δα = {width_alpha_multi:.3f}")
print(f"\nInterpretation:")
print(f"  Monofractal: narrow spectrum (single scaling)")
print(f"  Multifractal: broad spectrum (multiple scalings)")

## 5. Application to Neural Data

Analyze real/simulated neural time series.

In [ ]:
# Simulate "neural" signal (BOLD-like autocorrelated noise)
def generate_neural_signal(n, autocorr=0.95):
    """Generate autocorrelated signal similar to fMRI."""
    signal = np.zeros(n)
    signal[0] = np.random.randn()
    
    for i in range(1, n):
        signal[i] = autocorr * signal[i-1] + np.sqrt(1 - autocorr**2) * np.random.randn()
    
    # Add multiplicative noise (creates multifractality)
    envelope = np.abs(signal)
    signal *= (1 + 0.3 * envelope / envelope.max())
    
    return signal

neural_signal = generate_neural_signal(4096)

# Analyze
mfdfa_neural = MultifractalDetrendedFluctuationAnalysis(q_values=q_values, scales=scales)
result_neural = mfdfa_neural.analyze(neural_signal)

# Visualize signal
plt.figure(figsize=(14, 4))
plt.plot(neural_signal, 'k-', linewidth=0.5)
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.title('Simulated Neural Signal')
plt.grid(True, alpha=0.3)
plt.show()

# MF-DFA results
h_neural = result_neural['hurst_exponents']
tau_neural = q_values * h_neural - 1
alpha_neural, f_alpha_neural = compute_singularity_spectrum(q_values, tau_neural)

# Summary plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.plot(q_values, h_neural, 'go-', linewidth=2, markersize=8)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Moment q')
plt.ylabel('h(q)')
plt.title('Generalized Hurst Exponent')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(alpha_neural, f_alpha_neural, 'go-', linewidth=2, markersize=8)
plt.xlabel('Hölder Exponent α')
plt.ylabel('f(α)')
plt.title('Singularity Spectrum')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
width_neural = alpha_neural.max() - alpha_neural.min()
asymmetry = (alpha_neural.max() - alpha_neural[len(alpha_neural)//2]) - \
            (alpha_neural[len(alpha_neural)//2] - alpha_neural.min())
plt.bar(['Width Δα', 'Asymmetry'], [width_neural, asymmetry], 
        color=['green', 'orange'], alpha=0.7)
plt.ylabel('Value')
plt.title('Multifractal Metrics')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Statistical test for multifractality
# Shuffle phases to destroy multifractal structure
fft_signal = np.fft.fft(neural_signal)
phases = np.angle(fft_signal)
shuffled_phases = np.random.permutation(phases)
surrogate = np.fft.ifft(np.abs(fft_signal) * np.exp(1j * shuffled_phases)).real

result_surrogate = mfdfa_neural.analyze(surrogate)
h_surrogate = result_surrogate['hurst_exponents']
width_surrogate = h_surrogate.max() - h_surrogate.min()

print(f"\nMultifractality Test:")
print(f"Neural signal: Δα = {width_neural:.3f}")
print(f"Phase-shuffled: Δα = {width_surrogate:.3f}")
print(f"\nConclusion: Signal is {'significantly' if width_neural > 1.5*width_surrogate else 'not'} multifractal")

## Conclusion

This notebook demonstrated:
1. **MF-DFA**: Comprehensive multifractal characterization
2. **Generalized Hurst exponents**: h(q) across moments
3. **Singularity spectrum**: f(α) reveals scaling diversity
4. **Multifractal width**: Δα quantifies complexity
5. **Statistical testing**: Phase shuffling to test significance

Multifractal analysis reveals temporal complexity beyond simple scaling. In neuroscience, multifractality indicates rich temporal organization and adaptive information processing. Foundation models exhibiting multifractal dynamics may capture brain-like temporal structure.